# CrimeScope ML — 04. Train & Evaluate v3 (Lakehouse + MLflow)

**Description:** Optuna-tuned LightGBM with sqrt+log1p ensemble, regularization,
sample weighting, feature pruning, weighted baseline, violent/property sub-models,
SHAP explanations, MLflow tracking, Unity Catalog registry.

**v3 improvements:**
- Optuna hyperparameter tuning (20 trials)
- Dual-transform ensemble: sqrt + log1p averaged
- L1/L2 regularization for SHAP diversity
- Sample weighting for high-crime tracts
- Feature pruning (drop bottom 25% importance)

**Prerequisites:** `03` (feature table `tract_crime_features` with violent/property split).

In [ ]:
%pip install lightgbm shap optuna "mlflow==2.14.3" "opentelemetry-sdk==1.22.0" "opentelemetry-api==1.22.0" "opentelemetry-semantic-conventions==0.43b0" -q

In [ ]:
import json

import lightgbm as lgb
import mlflow
import numpy as np
import optuna
import pandas as pd
from pyspark.sql import functions as F
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")

FEATURES_TABLE = "varanasi.default.tract_crime_features"
TEST_MONTHS = 6
EXPERIMENT = "/Shared/Team_varanasi/crimescope_v3"

FEATURE_COLS = [
    "lag_1m_count", "lag_2m_count", "lag_3m_count", "lag_6m_count", "lag_12m_count",
    "rolling_mean_3m", "rolling_mean_6m", "rolling_mean_12m",
    "rolling_std_6m", "rolling_max_6m", "rolling_min_6m",
    "rolling_std_12m", "rolling_max_12m", "rolling_min_12m",
    "mom_change",
    "violent_lag_1m", "violent_lag_3m", "violent_lag_6m",
    "violent_rolling_3m", "violent_rolling_6m", "violent_rolling_12m",
    "violent_ratio", "violent_ratio_6m",
    "property_lag_1m", "property_lag_3m", "property_lag_6m",
    "property_rolling_3m", "property_rolling_6m", "property_rolling_12m",
    "month_of_year", "month_sin", "month_cos", "year",
    "same_month_last_year", "yoy_change",
    "total_pop_acs", "median_hh_income_acs", "poverty_rate_acs", "housing_units_acs",
    "log_pop", "log_income",
    "crime_rate_per_1k", "poverty_x_crime", "income_crime_ratio", "pop_per_housing_unit",
    "city_month_total", "city_total_lag1", "tract_share_of_city",
    "tract_vs_city_avg", "ca_avg_crime", "tract_vs_ca_avg",
    "trend_accel", "cv_6m", "cv_12m", "trend_3m",
]

print(f"Features: {len(FEATURE_COLS)}")

---
## 1. Load features and time-based split

In [ ]:
sdf = spark.table(FEATURES_TABLE)
print(f"Total rows in {FEATURES_TABLE}: {sdf.count():,}")

sdf = sdf.filter(F.col("lag_1m_count").isNotNull())
sdf = sdf.filter(F.col("y_next_30d_count").isNotNull())

for col in FEATURE_COLS:
    sdf = sdf.withColumn(col, F.coalesce(F.col(col), F.lit(0.0)))

n_usable = sdf.count()
print(f"Usable rows: {n_usable:,}")

max_m = sdf.select(F.max("month_start").alias("m")).first()["m"]
min_m = sdf.select(F.min("month_start").alias("m")).first()["m"]
n_months = (max_m.year - min_m.year) * 12 + (max_m.month - min_m.month) + 1
actual_test_months = min(TEST_MONTHS, max(1, n_months // 3))

test_start = sdf.select(F.add_months(F.lit(max_m), -(actual_test_months - 1)).alias("c")).first()["c"]

select_cols = FEATURE_COLS + ["y_next_30d_count", "y_next_30d_violent", "y_next_30d_property",
                              "violent_count", "property_count",
                              "tract_geoid", "month_start"]

train_sdf = sdf.filter(F.col("month_start") < test_start)
test_sdf = sdf.filter(F.col("month_start") >= test_start)

train_pdf = train_sdf.select(*select_cols).toPandas()
test_pdf = test_sdf.select(*select_cols).toPandas()

X_train = train_pdf[FEATURE_COLS].astype(float).fillna(0)
y_train = train_pdf["y_next_30d_count"].astype(float)
X_test = test_pdf[FEATURE_COLS].astype(float).fillna(0)
y_test = test_pdf["y_next_30d_count"].astype(float)

y_train_v = train_pdf["y_next_30d_violent"].astype(float).fillna(0)
y_test_v = test_pdf["y_next_30d_violent"].astype(float).fillna(0)
y_train_p = train_pdf["y_next_30d_property"].astype(float).fillna(0)
y_test_p = test_pdf["y_next_30d_property"].astype(float).fillna(0)

# Sample weights: higher weight for high-crime tracts
train_weights = np.log1p(y_train) + 1.0

print(f"\nData: {n_months} months ({min_m} to {max_m})")
print(f"Test window: >= {test_start} ({actual_test_months} months)")
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

---
## 2. Weighted rules baseline

In [ ]:
def weighted_baseline(X):
    pred = (
        0.30 * X["rolling_mean_3m"].fillna(0) +
        0.25 * X["rolling_mean_12m"].fillna(0) +
        0.20 * X["lag_1m_count"].fillna(0) +
        0.15 * X["same_month_last_year"].fillna(X["rolling_mean_12m"].fillna(0)) +
        0.10 * (X["city_month_total"].fillna(0) * X["tract_share_of_city"].fillna(0))
    )
    return np.maximum(pred.values, 0)

baseline_pred = weighted_baseline(X_test)
naive_pred = X_test["lag_1m_count"].values

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

naive_mae = mean_absolute_error(y_test, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
naive_r2 = r2_score(y_test, naive_pred)

mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name="weighted_rules_baseline"):
    mlflow.log_param("model_type", "weighted_rules_index")
    mlflow.log_metric("mae", float(baseline_mae))
    mlflow.log_metric("rmse", float(baseline_rmse))
    mlflow.log_metric("r2", float(baseline_r2))

print(f"Naive lag-1       \u2014 MAE: {naive_mae:.4f}  |  RMSE: {naive_rmse:.4f}  |  R\u00b2: {naive_r2:.4f}")
print(f"Weighted baseline \u2014 MAE: {baseline_mae:.4f}  |  RMSE: {baseline_rmse:.4f}  |  R\u00b2: {baseline_r2:.4f}")

---
## 3. Optuna hyperparameter tuning

In [ ]:
def objective(trial):
    p = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "random_state": 42,
        "n_estimators": 500,
        "num_leaves": trial.suggest_int("num_leaves", 31, 127),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    }
    m = lgb.LGBMRegressor(**p)
    m.fit(
        X_train, np.log1p(y_train),
        sample_weight=train_weights,
        eval_set=[(X_test, np.log1p(y_test))],
        callbacks=[lgb.early_stopping(30, verbose=False)],
    )
    pred = np.expm1(np.maximum(m.predict(X_test), 0))
    pred = np.maximum(pred, 0)
    return mean_absolute_error(y_test, pred)

study = optuna.create_study(direction="minimize", study_name="crimescope_v3")
study.optimize(objective, n_trials=20, show_progress_bar=True)

best = study.best_params
print(f"\nBest MAE: {study.best_value:.4f}")
print(f"Best params: {json.dumps(best, indent=2)}")

---
## 4. Train ensemble: log1p model + sqrt model (tuned params)

In [ ]:
tuned_params = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "random_state": 42,
    "n_estimators": 500,
    **best,
}

# Model A: log1p transform (good for low-crime tracts)
model_log = lgb.LGBMRegressor(**tuned_params)
model_log.fit(
    X_train, np.log1p(y_train),
    sample_weight=train_weights,
    eval_set=[(X_test, np.log1p(y_test))],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)
print(f"Log1p model \u2014 best iteration: {model_log.best_iteration_}")

# Model B: sqrt transform (better for high-crime tracts)
model_sqrt = lgb.LGBMRegressor(**tuned_params)
model_sqrt.fit(
    X_train, np.sqrt(y_train),
    sample_weight=train_weights,
    eval_set=[(X_test, np.sqrt(y_test))],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)
print(f"Sqrt model  \u2014 best iteration: {model_sqrt.best_iteration_}")

---
## 5. Evaluate ensemble vs individual models

In [ ]:
pred_log = np.maximum(np.expm1(np.maximum(model_log.predict(X_test), 0)), 0)
pred_sqrt = np.maximum(np.square(np.maximum(model_sqrt.predict(X_test), 0)), 0)
pred_ensemble = 0.5 * pred_log + 0.5 * pred_sqrt

def calc_metrics(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

mae_log, rmse_log, r2_log = calc_metrics("log1p", y_test, pred_log)
mae_sqrt, rmse_sqrt, r2_sqrt = calc_metrics("sqrt", y_test, pred_sqrt)
mae_ens, rmse_ens, r2_ens = calc_metrics("ensemble", y_test, pred_ensemble)

# Pick best overall prediction
if mae_ens <= min(mae_log, mae_sqrt):
    y_pred = pred_ensemble
    mae, rmse, r2 = mae_ens, rmse_ens, r2_ens
    best_strategy = "ensemble"
elif mae_log <= mae_sqrt:
    y_pred = pred_log
    mae, rmse, r2 = mae_log, rmse_log, r2_log
    best_strategy = "log1p"
else:
    y_pred = pred_sqrt
    mae, rmse, r2 = mae_sqrt, rmse_sqrt, r2_sqrt
    best_strategy = "sqrt"

mae_improve = (1 - mae / baseline_mae) * 100
rmse_improve = (1 - rmse / baseline_rmse) * 100

print(f"{'='*70}")
print(f"{'Model':<22} {'MAE':>8}  {'RMSE':>8}  {'R\u00b2':>8}")
print(f"{'='*70}")
print(f"{'Naive lag-1':<22} {naive_mae:>8.4f}  {naive_rmse:>8.4f}  {naive_r2:>8.4f}")
print(f"{'Weighted baseline':<22} {baseline_mae:>8.4f}  {baseline_rmse:>8.4f}  {baseline_r2:>8.4f}")
print(f"{'LightGBM (log1p)':<22} {mae_log:>8.4f}  {rmse_log:>8.4f}  {r2_log:>8.4f}")
print(f"{'LightGBM (sqrt)':<22} {mae_sqrt:>8.4f}  {rmse_sqrt:>8.4f}  {r2_sqrt:>8.4f}")
print(f"{'LightGBM (ensemble)':<22} {mae_ens:>8.4f}  {rmse_ens:>8.4f}  {r2_ens:>8.4f}  \u2190")
print(f"{'='*70}")
print(f"Best strategy: {best_strategy}")
print(f"ML vs Weighted \u2014 MAE: {mae_improve:+.1f}%  |  RMSE: {rmse_improve:+.1f}%")
print(f"ML vs Naive    \u2014 MAE: {(1-mae/naive_mae)*100:+.1f}%")

# Error by magnitude (check high-crime tract improvement)
test_pdf["pred_ensemble"] = y_pred
test_pdf["abs_error"] = np.abs(y_test.values - y_pred)
for tier_name, lo, hi in [("High crime (>50)", 50, 9999), ("Medium (10-50)", 10, 50), ("Low (<10)", 0, 10)]:
    mask = (y_test >= lo) & (y_test < hi)
    if mask.sum() > 0:
        tier_mae = mean_absolute_error(y_test[mask], y_pred[mask])
        print(f"  {tier_name}: n={mask.sum():,}, MAE={tier_mae:.3f}")

---
## 6. Feature pruning — identify and remove dead-weight features

In [ ]:
imp_vals = model_log.feature_importances_
threshold = np.percentile(imp_vals, 25)
keep_mask = imp_vals > threshold

FEATURE_COLS_PRUNED = [f for f, k in zip(FEATURE_COLS, keep_mask) if k]
dropped = [f for f, k in zip(FEATURE_COLS, keep_mask) if not k]

print(f"Feature pruning: {len(FEATURE_COLS)} \u2192 {len(FEATURE_COLS_PRUNED)} features")
print(f"Dropped ({len(dropped)}): {dropped}")
print(f"Kept ({len(FEATURE_COLS_PRUNED)}): {FEATURE_COLS_PRUNED}")

---
## 7. Train violent + property sub-models (pruned features, tuned params)

In [ ]:
X_train_p = X_train[FEATURE_COLS_PRUNED]
X_test_p = X_test[FEATURE_COLS_PRUNED]

weights_v = np.log1p(y_train_v) + 1.0
weights_p = np.log1p(y_train_p) + 1.0

model_violent = lgb.LGBMRegressor(**tuned_params)
model_violent.fit(
    X_train_p, np.log1p(y_train_v),
    sample_weight=weights_v,
    eval_set=[(X_test_p, np.log1p(y_test_v))],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)

model_property = lgb.LGBMRegressor(**tuned_params)
model_property.fit(
    X_train_p, np.log1p(y_train_p),
    sample_weight=weights_p,
    eval_set=[(X_test_p, np.log1p(y_test_p))],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
)

y_pred_v = np.maximum(np.expm1(np.maximum(model_violent.predict(X_test_p), 0)), 0)
y_pred_p = np.maximum(np.expm1(np.maximum(model_property.predict(X_test_p), 0)), 0)

mae_v, rmse_v, r2_v = calc_metrics("violent", y_test_v, y_pred_v)
mae_p, rmse_p, r2_p = calc_metrics("property", y_test_p, y_pred_p)

print(f"Violent  \u2014 MAE: {mae_v:.4f}  RMSE: {rmse_v:.4f}  R\u00b2: {r2_v:.4f}  (best_iter: {model_violent.best_iteration_})")
print(f"Property \u2014 MAE: {mae_p:.4f}  RMSE: {rmse_p:.4f}  R\u00b2: {r2_p:.4f}  (best_iter: {model_property.best_iteration_})")

---
## 8. Feature importance + SHAP

In [ ]:
imp = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance": model_log.feature_importances_,
}).sort_values("importance", ascending=False)

display(imp)

In [ ]:
import shap
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(model_log)
shap_sample = X_test.sample(n=min(2000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(shap_sample)

fig_shap, _ = plt.subplots(figsize=(10, 12))
shap.summary_plot(shap_values, shap_sample, show=False, max_display=25)
plt.tight_layout()
shap_path = "/tmp/shap_summary.png"
plt.savefig(shap_path, dpi=150, bbox_inches="tight")
plt.close()

fig_bar, _ = plt.subplots(figsize=(10, 12))
shap.summary_plot(shap_values, shap_sample, plot_type="bar", show=False, max_display=25)
plt.tight_layout()
shap_bar_path = "/tmp/shap_bar.png"
plt.savefig(shap_bar_path, dpi=150, bbox_inches="tight")
plt.close()

# SHAP diversity check
top1 = pd.Series([FEATURE_COLS[np.abs(shap_values[i]).argmax()] for i in range(len(shap_values))])
print(f"SHAP top-1 diversity ({top1.nunique()} unique):")
print(top1.value_counts().head(10))

display(fig_shap)
display(fig_bar)

---
## 9. Evaluation plots

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig1, ax1 = plt.subplots(figsize=(7, 7))
ax1.scatter(y_test, y_pred, alpha=0.15, s=8, color="#2563eb")
lim = max(y_test.max(), max(y_pred)) * 1.05
ax1.plot([0, lim], [0, lim], "--", color="#ef4444", linewidth=1.5, label="Perfect")
ax1.set_xlabel("Actual")
ax1.set_ylabel("Predicted")
ax1.set_title(f"Actual vs Predicted (ensemble)  |  MAE={mae:.2f}  R\u00b2={r2:.3f}")
ax1.legend()
fig1.tight_layout()
scatter_path = "/tmp/actual_vs_predicted.png"
fig1.savefig(scatter_path, dpi=150)
display(fig1)

fig2, ax2 = plt.subplots(figsize=(10, 12))
imp_sorted = imp.sort_values("importance", ascending=True)
ax2.barh(imp_sorted["feature"], imp_sorted["importance"], color="#2563eb")
ax2.set_xlabel("Importance (split count)")
ax2.set_title("Feature Importance \u2014 LightGBM (log1p model)")
fig2.tight_layout()
imp_path = "/tmp/feature_importance.png"
fig2.savefig(imp_path, dpi=150)
display(fig2)

residuals = y_test.values - y_pred
fig3, ax3 = plt.subplots(figsize=(8, 5))
ax3.hist(residuals, bins=80, color="#2563eb", alpha=0.7, edgecolor="white")
ax3.axvline(0, color="#ef4444", linestyle="--", linewidth=1.5)
ax3.set_xlabel("Residual (Actual \u2212 Predicted)")
ax3.set_ylabel("Count")
ax3.set_title("Residual Distribution")
fig3.tight_layout()
resid_path = "/tmp/residual_distribution.png"
fig3.savefig(resid_path, dpi=150)
display(fig3)

fig4, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, yt, yp, m, r) in zip(axes, [
    ("Overall (ensemble)", y_test, y_pred, mae, r2),
    ("Violent", y_test_v, y_pred_v, mae_v, r2_v),
    ("Property", y_test_p, y_pred_p, mae_p, r2_p),
]):
    ax.scatter(yt, yp, alpha=0.1, s=6, color="#2563eb")
    lim = max(yt.max(), max(yp)) * 1.05
    ax.plot([0, lim], [0, lim], "--", color="#ef4444", linewidth=1)
    ax.set_title(f"{name}  MAE={m:.2f}  R\u00b2={r:.3f}", fontsize=10)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
fig4.suptitle("All Three Models: Actual vs Predicted", fontsize=12)
fig4.tight_layout()
three_models_path = "/tmp/three_models.png"
fig4.savefig(three_models_path, dpi=150)
display(fig4)

plt.close("all")
print("All evaluation plots generated")

---
## 10. Log to MLflow + register models in Unity Catalog

In [ ]:
from mlflow.models.signature import infer_signature

mlflow.set_experiment(EXPERIMENT)
mlflow.set_registry_uri("databricks-uc")

MODEL_NAME = "varanasi.default.crimescope_risk_model"
MODEL_NAME_SQRT = "varanasi.default.crimescope_risk_model_sqrt"
MODEL_NAME_V = "varanasi.default.crimescope_violent_model"
MODEL_NAME_P = "varanasi.default.crimescope_property_model"

def register_model(model_obj, name, run_name, extra_metrics=None, features=None):
    feat_cols = features or FEATURE_COLS
    X_sample = X_train[feat_cols].head(5)
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({k: v for k, v in tuned_params.items() if k not in ('objective','metric','verbosity','random_state')})
        mlflow.log_param("n_features", len(feat_cols))
        mlflow.log_param("best_iteration", model_obj.best_iteration_)
        if extra_metrics:
            for k, v in extra_metrics.items():
                mlflow.log_metric(k, float(v))
        sig = infer_signature(X_sample, model_obj.predict(X_sample))
        mlflow.lightgbm.log_model(model_obj, artifact_path="model", signature=sig, registered_model_name=name)
        rid = run.info.run_id
    client = mlflow.MlflowClient()
    mv = client.search_model_versions(f"name='{name}'")
    ver = max(int(v.version) for v in mv)
    client.set_registered_model_alias(name, "champion", str(ver))
    print(f"\u2705 {name} v{ver} (alias=champion)")

# Log overall (log1p) model with full artifacts
with mlflow.start_run(run_name="lightgbm_ensemble_v3") as run:
    mlflow.log_params({k: v for k, v in tuned_params.items() if k not in ('objective','metric','verbosity','random_state')})
    mlflow.log_param("features_table", FEATURES_TABLE)
    mlflow.log_param("target", "y_next_30d_count")
    mlflow.log_param("ensemble_strategy", best_strategy)
    mlflow.log_param("n_features", len(FEATURE_COLS))
    mlflow.log_param("n_features_pruned", len(FEATURE_COLS_PRUNED))
    mlflow.log_param("optuna_trials", 20)
    mlflow.log_param("sample_weighting", "log1p(y)+1")
    mlflow.log_metric("mae", float(mae))
    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("r2", float(r2))
    mlflow.log_metric("mae_log", float(mae_log))
    mlflow.log_metric("mae_sqrt", float(mae_sqrt))
    mlflow.log_metric("mae_ensemble", float(mae_ens))
    mlflow.log_metric("baseline_mae", float(baseline_mae))
    mlflow.log_metric("baseline_rmse", float(baseline_rmse))
    mlflow.log_metric("baseline_r2", float(baseline_r2))
    mlflow.log_metric("mae_improvement_pct", float(mae_improve))
    mlflow.log_metric("rmse_improvement_pct", float(rmse_improve))
    mlflow.log_metric("naive_mae", float(naive_mae))
    mlflow.log_metric("violent_mae", float(mae_v))
    mlflow.log_metric("violent_r2", float(r2_v))
    mlflow.log_metric("property_mae", float(mae_p))
    mlflow.log_metric("property_r2", float(r2_p))
    mlflow.log_metric("n_train", len(X_train))
    mlflow.log_metric("n_test", len(X_test))

    for path in [scatter_path, imp_path, resid_path, three_models_path]:
        mlflow.log_artifact(path, artifact_path="plots")
    for path in [shap_path, shap_bar_path]:
        mlflow.log_artifact(path, artifact_path="shap")

    imp.to_csv("/tmp/feature_importance.csv", index=False)
    mlflow.log_artifact("/tmp/feature_importance.csv")
    with open("/tmp/feature_list.json", "w") as f:
        json.dump(FEATURE_COLS, f, indent=2)
    mlflow.log_artifact("/tmp/feature_list.json")
    with open("/tmp/feature_list_pruned.json", "w") as f:
        json.dump(FEATURE_COLS_PRUNED, f, indent=2)
    mlflow.log_artifact("/tmp/feature_list_pruned.json")
    with open("/tmp/optuna_best_params.json", "w") as f:
        json.dump(best, f, indent=2)
    mlflow.log_artifact("/tmp/optuna_best_params.json")

    sig = infer_signature(X_train.head(5), model_log.predict(X_train.head(5)))
    mlflow.lightgbm.log_model(model_log, artifact_path="model", signature=sig, registered_model_name=MODEL_NAME)
    run_id = run.info.run_id

client = mlflow.MlflowClient()
mv = client.search_model_versions(f"name='{MODEL_NAME}'")
ver = max(int(v.version) for v in mv)
client.set_registered_model_alias(MODEL_NAME, "champion", str(ver))
print(f"\u2705 {MODEL_NAME} v{ver} (alias=champion, log1p model)")

# Register sqrt model
register_model(model_sqrt, MODEL_NAME_SQRT, "lightgbm_sqrt_v3",
               {"mae": mae_sqrt, "rmse": rmse_sqrt, "r2": r2_sqrt})

# Register violent and property sub-models (pruned features)
register_model(model_violent, MODEL_NAME_V, "lightgbm_violent_v3",
               {"mae": mae_v, "rmse": rmse_v, "r2": r2_v}, FEATURE_COLS_PRUNED)
register_model(model_property, MODEL_NAME_P, "lightgbm_property_v3",
               {"mae": mae_p, "rmse": rmse_p, "r2": r2_p}, FEATURE_COLS_PRUNED)

---
## 11. Save pruned feature list for notebook 05

In [ ]:
config = {
    "feature_cols": FEATURE_COLS,
    "feature_cols_pruned": FEATURE_COLS_PRUNED,
    "best_strategy": best_strategy,
    "tuned_params": {k: v for k, v in tuned_params.items() if k not in ('objective','metric','verbosity','random_state')},
    "model_names": {
        "overall_log": MODEL_NAME,
        "overall_sqrt": MODEL_NAME_SQRT,
        "violent": MODEL_NAME_V,
        "property": MODEL_NAME_P,
    },
}
VOLUME_PATH = "/Volumes/varanasi/default/ml_data"
with open(f"{VOLUME_PATH}/model_config_v3.json", "w") as f:
    json.dump(config, f, indent=2)
print(f"Saved model config to {VOLUME_PATH}/model_config_v3.json")
print(json.dumps(config, indent=2))

---
## 12. Summary

v3 pipeline has:

1. **Optuna-tuned hyperparameters** (20 trials with sample weighting)
2. **Dual-transform ensemble** (log1p + sqrt averaged)
3. **L1/L2 regularization** for SHAP diversity
4. **Sample weighting** for high-crime tract accuracy
5. **Feature pruning** (dropped bottom 25% importance)
6. **4 registered models** in Unity Catalog (log1p, sqrt, violent, property)

**Next:** Run notebook `05` to score all tracts using the ensemble.